In [1]:
import pandas as pd

# Load the newly uploaded SARIMA file
sarima = pd.read_csv("/content/sarima_residuals (2).csv")

print("Shape:", sarima.shape)
print("Columns:", sarima.columns.tolist())
print("\nFirst 5 rows:")
print(sarima.head())

# Check October 2024 specifically
sarima["month"] = pd.to_datetime(sarima["month"])

oct_2024 = sarima[sarima["month"] == "2024-10-01"]

print("\nOctober 2024 SARIMA residual:")
print(oct_2024)

Shape: (47, 2)
Columns: ['month', 'sarima_resid_z']

First 5 rows:
        month  sarima_resid_z
0  2022-02-01       -0.942194
1  2022-03-01        0.878326
2  2022-04-01        1.098041
3  2022-05-01        1.908578
4  2022-06-01       -0.452779

October 2024 SARIMA residual:
        month  sarima_resid_z
32 2024-10-01        2.119149


In [2]:
import pandas as pd

# Load the three authoritative source files
upi = pd.read_csv("/content/UPI_Master_Dataset (1).csv")
rho = pd.read_csv("/content/mm1_queuing_results.csv")
sarima = pd.read_csv("/content/sarima_residuals (2).csv")

print("=== UPI MASTER DATASET ===")
print("Shape:", upi.shape)
print("Columns:", upi.columns.tolist())
print("Date range:", upi["month"].min(), "to", upi["month"].max())

print("\n=== M/M/1 QUEUING RESULTS ===")
print("Shape:", rho.shape)
print("Columns:", rho.columns.tolist())

print("\n=== CORRECTED SARIMA ===")
print("Shape:", sarima.shape)
print("Columns:", sarima.columns.tolist())
print("Date range:", sarima["month"].min(), "to", sarima["month"].max())

# Confirm the critical corrected SARIMA value again
sarima["month"] = pd.to_datetime(sarima["month"])
print("\nOctober 2024 SARIMA residual:",
      sarima.loc[sarima["month"] == "2024-10-01", "sarima_resid_z"].iloc[0])

=== UPI MASTER DATASET ===
Shape: (125, 8)
Columns: ['month', 'upi_remitter_banks', 'total_volume_in_mn', 'td_pct', 'iss_mean_z', 'iss_max_z', 'stress_flag', 'max_bank']
Date range: 2024-01 to 2026-01

=== M/M/1 QUEUING RESULTS ===
Shape: (66, 11)
Columns: ['month', 'volume_mn', 'value_cr', 'days_in_month', 'seconds_in_month', 'total_transactions', 'lambda_avg_tps', 'rho_floor', 'rho_sens_low', 'rho_sens_high', 'rho_peak_baseline']

=== CORRECTED SARIMA ===
Shape: (47, 2)
Columns: ['month', 'sarima_resid_z']
Date range: 2022-02-01 to 2025-12-01

October 2024 SARIMA residual: 2.1191485423632934


In [3]:
import pandas as pd

# Load the three verified source files
upi = pd.read_csv("/content/UPI_Master_Dataset (1).csv")
rho = pd.read_csv("/content/mm1_queuing_results.csv")
sarima = pd.read_csv("/content/sarima_residuals (2).csv")

# Convert dates to monthly datetime format
upi["month"] = pd.to_datetime(upi["month"].astype(str))
rho["month"] = pd.to_datetime(rho["month"].astype(str))
sarima["month"] = pd.to_datetime(sarima["month"])

# ISS: one row per month from the authoritative master dataset
iss_monthly = (
    upi.groupby("month", as_index=False)
       .agg(
           iss_mean_z=("iss_mean_z", "mean"),
           iss_max_z=("iss_max_z", "mean")
       )
)

# Keep only the M/M/1 rho required for AMPI
rho_monthly = rho[["month", "rho_peak_baseline"]].copy()

# Merge the three components
ampi_input = (
    iss_monthly
    .merge(sarima[["month", "sarima_resid_z"]], on="month", how="inner")
    .merge(rho_monthly, on="month", how="inner")
)

# Restrict to the final common 24-month window
ampi_input = ampi_input[
    (ampi_input["month"] >= "2024-01-01") &
    (ampi_input["month"] <= "2025-12-01")
].sort_values("month").reset_index(drop=True)

print("AMPI input shape:", ampi_input.shape)
print("\nDate range:",
      ampi_input["month"].min().date(),
      "to",
      ampi_input["month"].max().date())

print("\nOctober 2024 check:")
print(
    ampi_input.loc[
        ampi_input["month"] == "2024-10-01"
    ]
)

print("\nMissing values:")
print(ampi_input.isna().sum())

print("\nFirst 5 rows:")
print(ampi_input.head())

# Save refreshed AMPI input
ampi_input.to_csv(
    "/content/AMPI_24_month_input_CORRECTED_SARIMA.csv",
    index=False
)

print("\nSaved:")
print("/content/AMPI_24_month_input_CORRECTED_SARIMA.csv")

AMPI input shape: (24, 5)

Date range: 2024-01-01 to 2025-12-01

October 2024 check:
       month  iss_mean_z  iss_max_z  sarima_resid_z  rho_peak_baseline
9 2024-10-01   -0.594233  -0.543342        2.119149           0.706547

Missing values:
month                0
iss_mean_z           0
iss_max_z            0
sarima_resid_z       0
rho_peak_baseline    0
dtype: int64

First 5 rows:
       month  iss_mean_z  iss_max_z  sarima_resid_z  rho_peak_baseline
0 2024-01-01    0.165913   1.330143        0.041450           0.519869
1 2024-02-01    1.334491   2.376912        1.006267           0.551152
2 2024-03-01    0.628053   1.823902        1.047107           0.572566
3 2024-04-01    2.889914   3.379372       -0.710446           0.585664
4 2024-05-01    0.397366   1.738498        0.150555           0.597950

Saved:
/content/AMPI_24_month_input_CORRECTED_SARIMA.csv


In [4]:
import numpy as np
import pandas as pd
from statsmodels.nonparametric.smoothers_lowess import lowess

# Work on the full M/M/1 rho series first
rho_full = rho[["month", "rho_peak_baseline"]].copy()
rho_full = rho_full.sort_values("month").reset_index(drop=True)

# Convert month to numeric time index
rho_full["time_index"] = np.arange(len(rho_full))

# LOWESS local trend
lowess_fit = lowess(
    endog=rho_full["rho_peak_baseline"],
    exog=rho_full["time_index"],
    frac=0.30,
    return_sorted=False
)

# De-trended rho = actual rho - local trend
rho_full["rho_detrended"] = (
    rho_full["rho_peak_baseline"] - lowess_fit
)

print("Full rho series:", rho_full.shape)
print("Date range:",
      rho_full["month"].min().date(),
      "to",
      rho_full["month"].max().date())

print("\nDe-trended rho statistics:")
print(rho_full["rho_detrended"].describe())

# Check the 2024–2025 period
rho_check = rho_full[
    (rho_full["month"] >= "2024-01-01") &
    (rho_full["month"] <= "2025-12-01")
].copy()

print("\n2024–2025 de-trended rho:")
print(
    rho_check[
        ["month", "rho_peak_baseline", "rho_detrended"]
    ].to_string(index=False)
)

# Save diagnostic file
rho_full.to_csv(
    "/content/rho_detrended_diagnostic_CORRECTED.csv",
    index=False
)

print("\nSaved:")
print("/content/rho_detrended_diagnostic_CORRECTED.csv")

Full rho series: (66, 4)
Date range: 2021-01-01 to 2026-06-01

De-trended rho statistics:
count    66.000000
mean     -0.000364
std       0.007684
min      -0.019920
25%      -0.005124
50%      -0.000327
75%       0.003161
max       0.028583
Name: rho_detrended, dtype: float64

2024–2025 de-trended rho:
     month  rho_peak_baseline  rho_detrended
2024-01-01           0.519869      -0.010883
2024-02-01           0.551152       0.004210
2024-03-01           0.572566       0.009576
2024-04-01           0.585664       0.006702
2024-05-01           0.597950       0.002950
2024-06-01           0.611247       0.000074
2024-07-01           0.614978      -0.012583
2024-08-01           0.637451      -0.006718
2024-09-01           0.662163       0.001155
2024-10-01           0.706547       0.028583
2024-11-01           0.681545      -0.013403
2024-12-01           0.712726       0.000736
2025-01-01           0.724058      -0.005148
2025-02-01           0.759666       0.013072
2025-03-01          

In [5]:
# Merge the corrected de-trended rho into the refreshed AMPI input

rho_detrended = rho_full[["month", "rho_detrended"]].copy()

ampi_input_corrected = (
    ampi_input
    .drop(columns=["rho_peak_baseline"])
    .merge(rho_detrended, on="month", how="inner")
    .sort_values("month")
    .reset_index(drop=True)
)

# Goalpost normalization: 70–130
def normalize_70_130(series):
    s_min = series.min()
    s_max = series.max()
    return 70 + 60 * ((series - s_min) / (s_max - s_min))

# Primary components
ampi_input_corrected["iss_mean_norm"] = normalize_70_130(
    ampi_input_corrected["iss_mean_z"]
)

ampi_input_corrected["sarima_norm"] = normalize_70_130(
    ampi_input_corrected["sarima_resid_z"]
)

ampi_input_corrected["rho_norm"] = normalize_70_130(
    ampi_input_corrected["rho_detrended"]
)

print("Shape:", ampi_input_corrected.shape)

print("\nNormalized component range:")
print(
    ampi_input_corrected[
        ["iss_mean_norm", "sarima_norm", "rho_norm"]
    ].agg(["min", "max"])
)

print("\nOctober 2024 normalized components:")
print(
    ampi_input_corrected.loc[
        ampi_input_corrected["month"] == "2024-10-01",
        [
            "month",
            "iss_mean_z",
            "sarima_resid_z",
            "rho_detrended",
            "iss_mean_norm",
            "sarima_norm",
            "rho_norm"
        ]
    ].to_string(index=False)
)

print("\nFirst 5 rows:")
print(
    ampi_input_corrected[
        [
            "month",
            "iss_mean_norm",
            "sarima_norm",
            "rho_norm"
        ]
    ].head()
)

Shape: (24, 8)

Normalized component range:
     iss_mean_norm  sarima_norm  rho_norm
min           70.0         70.0      70.0
max          130.0        130.0     130.0

October 2024 normalized components:
     month  iss_mean_z  sarima_resid_z  rho_detrended  iss_mean_norm  sarima_norm  rho_norm
2024-10-01   -0.594233        2.119149       0.028583      70.517691        130.0     130.0

First 5 rows:
       month  iss_mean_norm  sarima_norm    rho_norm
0 2024-01-01      83.495102   103.369885   73.601481
1 2024-02-01     103.445385   115.736065   95.169575
2 2024-03-01      91.384883   116.259516  102.837797
3 2024-04-01     130.000000    93.732736   98.731338
4 2024-05-01      87.446533   104.768291   93.369165


In [6]:
import pandas as pd

# ============================================================
# USE THE PREVIOUSLY VALIDATED RHO NORMALIZED VALUES
# ============================================================

validated_rho_norm = {
    "2024-01-01": 78.050377,
    "2024-02-01": 96.578449,
    "2024-03-01": 102.590156,
    "2024-04-01": 95.250129,
    "2024-05-01": 94.570154,
    "2024-06-01": 95.024801,
    "2024-07-01": 78.623843,
    "2024-08-01": 79.691250,
    "2024-09-01": 94.794019,
    "2024-10-01": 130.000000,
    "2024-11-01": 70.000000,
    "2024-12-01": 86.558907,
    "2025-01-01": 81.889347,
    "2025-02-01": 102.272226,
    "2025-03-01": 107.113836,
    "2025-04-01": 94.184759,
    "2025-05-01": 85.752115,
    "2025-06-01": 85.109340,
    "2025-07-01": 91.159084,
    "2025-08-01": 99.352647,
    "2025-09-01": 90.375683,
    "2025-10-01": 89.149580,
    "2025-11-01": 95.748654,
    "2025-12-01": 97.390092
}

# Convert dictionary to DataFrame
rho_validated = pd.DataFrame(
    list(validated_rho_norm.items()),
    columns=["month", "rho_norm"]
)

rho_validated["month"] = pd.to_datetime(rho_validated["month"])

# Merge with the CORRECTED SARIMA AMPI input
ampi_final_input = (
    ampi_input
    .drop(columns=["rho_peak_baseline"])
    .merge(rho_validated, on="month", how="inner")
    .sort_values("month")
    .reset_index(drop=True)
)

# Verify exactly 24 months
print("Shape:", ampi_final_input.shape)

print("\nDate range:",
      ampi_final_input["month"].min().date(),
      "to",
      ampi_final_input["month"].max().date())

print("\nMissing values:")
print(ampi_final_input.isna().sum())

# Critical verification points
print("\n=== VERIFICATION ===")

print("\nOctober 2024:")
print(
    ampi_final_input[
        ampi_final_input["month"] == "2024-10-01"
    ].to_string(index=False)
)

print("\nMarch 2025:")
print(
    ampi_final_input[
        ampi_final_input["month"] == "2025-03-01"
    ].to_string(index=False)
)

print("\nFirst 5 rows:")
print(ampi_final_input.head().to_string(index=False))

# Save refreshed AMPI input
output_path = "/content/AMPI_24_month_input_CORRECTED_FINAL.csv"

ampi_final_input.to_csv(output_path, index=False)

print("\nSaved:", output_path)

Shape: (24, 5)

Date range: 2024-01-01 to 2025-12-01

Missing values:
month             0
iss_mean_z        0
iss_max_z         0
sarima_resid_z    0
rho_norm          0
dtype: int64

=== VERIFICATION ===

October 2024:
     month  iss_mean_z  iss_max_z  sarima_resid_z  rho_norm
2024-10-01   -0.594233  -0.543342        2.119149     130.0

March 2025:
     month  iss_mean_z  iss_max_z  sarima_resid_z   rho_norm
2025-03-01   -0.059208   1.930034        1.497364 107.113836

First 5 rows:
     month  iss_mean_z  iss_max_z  sarima_resid_z   rho_norm
2024-01-01    0.165913   1.330143        0.041450  78.050377
2024-02-01    1.334491   2.376912        1.006267  96.578449
2024-03-01    0.628053   1.823902        1.047107 102.590156
2024-04-01    2.889914   3.379372       -0.710446  95.250129
2024-05-01    0.397366   1.738498        0.150555  94.570154

Saved: /content/AMPI_24_month_input_CORRECTED_FINAL.csv


In [7]:
import numpy as np

# ============================================================
# FINAL AMPI CALCULATION
# Uses the same validated methodology:
# Mean - CVPenalty
# SD uses ddof=1
# ============================================================

# Three primary normalized components
components = [
    "iss_mean_norm",
    "sarima_norm",
    "rho_norm"
]

# Add the validated normalization for ISS and SARIMA
def normalize_70_130(series):
    s_min = series.min()
    s_max = series.max()
    return 70 + 60 * ((series - s_min) / (s_max - s_min))

ampi_final = ampi_final_input.copy()

ampi_final["iss_mean_norm"] = normalize_70_130(
    ampi_final["iss_mean_z"]
)

ampi_final["sarima_norm"] = normalize_70_130(
    ampi_final["sarima_resid_z"]
)

# Mean of the three components
ampi_final["Mean"] = ampi_final[
    components
].mean(axis=1)

# Sample standard deviation
ampi_final["SD"] = ampi_final[
    components
].std(axis=1, ddof=1)

# Coefficient of variation
ampi_final["CV"] = (
    ampi_final["SD"] / ampi_final["Mean"]
)

# CV penalty
ampi_final["CVPenalty"] = (
    ampi_final["Mean"] * ampi_final["CV"]**2
)

# Final AMPI
ampi_final["AMPI"] = (
    ampi_final["Mean"] - ampi_final["CVPenalty"]
)

# ============================================================
# CHECK RESULTS
# ============================================================

print("Observations:", len(ampi_final))

print("\nAMPI summary:")
print(
    ampi_final["AMPI"].agg(
        ["min", "max", "mean", "median"]
    )
)

print("\nHighest AMPI:")
print(
    ampi_final.loc[
        ampi_final["AMPI"].idxmax(),
        ["month", "AMPI"]
    ]
)

print("\nLowest AMPI:")
print(
    ampi_final.loc[
        ampi_final["AMPI"].idxmin(),
        ["month", "AMPI"]
    ]
)

print("\nOctober 2024:")
print(
    ampi_final[
        ampi_final["month"] == "2024-10-01"
    ][
        [
            "month",
            "iss_mean_norm",
            "sarima_norm",
            "rho_norm",
            "Mean",
            "SD",
            "CV",
            "CVPenalty",
            "AMPI"
        ]
    ].to_string(index=False)
)

print("\nMarch 2025:")
print(
    ampi_final[
        ampi_final["month"] == "2025-03-01"
    ][
        [
            "month",
            "iss_mean_norm",
            "sarima_norm",
            "rho_norm",
            "Mean",
            "SD",
            "CV",
            "CVPenalty",
            "AMPI"
        ]
    ].to_string(index=False)
)

Observations: 24

AMPI summary:
min        75.380277
max       104.358268
mean       89.633781
median     88.584348
Name: AMPI, dtype: float64

Highest AMPI:
month    2024-02-01 00:00:00
AMPI              104.358268
Name: 1, dtype: object

Lowest AMPI:
month    2024-11-01 00:00:00
AMPI               75.380277
Name: 10, dtype: object

October 2024:
     month  iss_mean_norm  sarima_norm  rho_norm       Mean        SD       CV  CVPenalty      AMPI
2024-10-01      70.517691        130.0     130.0 110.172564 34.342127 0.311712  10.704858 99.467706

March 2025:
     month  iss_mean_norm  sarima_norm   rho_norm       Mean        SD       CV  CVPenalty      AMPI
2025-03-01      79.651774   122.030511 107.113836 102.932041 21.496625 0.208843   4.489418 98.442623


In [8]:
# ============================================================
# SAVE THE CORRECTED FINAL AMPI OUTPUTS
# ============================================================

# Full final AMPI results
final_ampi_path = "/content/AMPI_FINAL_CORRECTED_SARIMA.csv"
ampi_final.to_csv(final_ampi_path, index=False)

# Summary statistics
summary = pd.DataFrame({
    "metric": [
        "observations",
        "minimum",
        "maximum",
        "mean",
        "median",
        "highest_month",
        "highest_AMPI",
        "lowest_month",
        "lowest_AMPI"
    ],
    "value": [
        len(ampi_final),
        ampi_final["AMPI"].min(),
        ampi_final["AMPI"].max(),
        ampi_final["AMPI"].mean(),
        ampi_final["AMPI"].median(),
        ampi_final.loc[ampi_final["AMPI"].idxmax(), "month"].strftime("%Y-%m"),
        ampi_final["AMPI"].max(),
        ampi_final.loc[ampi_final["AMPI"].idxmin(), "month"].strftime("%Y-%m"),
        ampi_final["AMPI"].min()
    ]
})

summary_path = "/content/AMPI_FINAL_CORRECTED_SARIMA_SUMMARY.csv"
summary.to_csv(summary_path, index=False)

print("Saved final AMPI:")
print(final_ampi_path)

print("\nSaved summary:")
print(summary_path)

print("\nFinal AMPI columns:")
print(ampi_final.columns.tolist())

print("\nFinal 24-month AMPI:")
print(
    ampi_final[
        ["month", "iss_mean_norm", "sarima_norm",
         "rho_norm", "Mean", "SD", "CVPenalty", "AMPI"]
    ].to_string(index=False)
)

Saved final AMPI:
/content/AMPI_FINAL_CORRECTED_SARIMA.csv

Saved summary:
/content/AMPI_FINAL_CORRECTED_SARIMA_SUMMARY.csv

Final AMPI columns:
['month', 'iss_mean_z', 'iss_max_z', 'sarima_resid_z', 'rho_norm', 'iss_mean_norm', 'sarima_norm', 'Mean', 'SD', 'CV', 'CVPenalty', 'AMPI']

Final 24-month AMPI:
     month  iss_mean_norm  sarima_norm   rho_norm       Mean        SD  CVPenalty       AMPI
2024-01-01      83.495102   103.369885  78.050377  88.305121 13.327475   2.011453  86.293669
2024-02-01     103.445385   115.736065  96.578449 105.253300  9.705925   0.895031 104.358268
2024-03-01      91.384883   116.259516 102.590156 103.411518 12.457641   1.500731 101.910788
2024-04-01     130.000000    93.732736  95.250129 106.327622 20.514915   3.958160 102.369461
2024-05-01      87.446533   104.768291  94.570154  95.594992  8.706236   0.792913  94.802079
2024-06-01      74.829032    98.203755  95.024801  89.352530 12.677753   1.798779  87.553751
2024-07-01      76.219596    95.757964  78

In [9]:
import pandas as pd

# Load the authoritative ISS dataset
upi = pd.read_csv("/content/UPI_Master_Dataset (1).csv")

print("Shape:", upi.shape)
print("\nColumns:")
print(upi.columns.tolist())

print("\nUnique months:")
print(upi["month"].unique())

print("\nStress flag counts:")
print(upi["stress_flag"].value_counts(dropna=False))

print("\nStress-flagged rows:")
print(
    upi[
        upi["stress_flag"].astype(str).str.lower().isin(
            ["1", "true", "yes", "y"]
        )
    ][
        [
            "month",
            "upi_remitter_banks",
            "iss_mean_z",
            "iss_max_z",
            "stress_flag",
            "max_bank"
        ]
    ].to_string(index=False)
)

Shape: (125, 8)

Columns:
['month', 'upi_remitter_banks', 'total_volume_in_mn', 'td_pct', 'iss_mean_z', 'iss_max_z', 'stress_flag', 'max_bank']

Unique months:
['2024-01' '2024-02' '2024-03' '2024-04' '2024-05' '2024-06' '2024-07'
 '2024-08' '2024-09' '2024-10' '2024-11' '2024-12' '2025-01' '2025-02'
 '2025-03' '2025-04' '2025-05' '2025-06' '2025-07' '2025-08' '2025-09'
 '2025-10' '2025-11' '2025-12' '2026-01']

Stress flag counts:
stress_flag
False    120
True       5
Name: count, dtype: int64

Stress-flagged rows:
  month   upi_remitter_banks  iss_mean_z  iss_max_z  stress_flag       max_bank
2024-04  State Bank of India    2.889914   3.379372         True Bank of Baroda
2024-04       HDFC Bank Ltd.    2.889914   3.379372         True Bank of Baroda
2024-04       Bank of Baroda    2.889914   3.379372         True Bank of Baroda
2024-04  Union Bank of India    2.889914   3.379372         True Bank of Baroda
2024-04 Punjab National Bank    2.889914   3.379372         True Bank of Barod

In [10]:
import pandas as pd
import numpy as np

# Load the authoritative ISS dataset
upi = pd.read_csv("/content/UPI_Master_Dataset (1).csv")

# Keep one monthly ISS value
# The same monthly ISS value is repeated for the 5 banks,
# so take the first value for each month.
iss_monthly = (
    upi.groupby("month", as_index=False)
       .agg(
           iss_mean_z=("iss_mean_z", "first"),
           iss_max_z=("iss_max_z", "first")
       )
)

# Convert month to datetime
iss_monthly["month"] = pd.to_datetime(
    iss_monthly["month"].astype(str)
)

# Use the 24-month common analysis period
iss_monthly = iss_monthly[
    (iss_monthly["month"] >= "2024-01-01") &
    (iss_monthly["month"] <= "2025-12-01")
].sort_values("month").reset_index(drop=True)

# Define the documented March–May 2025
# disruption-and-response period
disruption_mask = (
    (iss_monthly["month"] >= "2025-03-01") &
    (iss_monthly["month"] <= "2025-05-01")
)

disruption = iss_monthly.loc[
    disruption_mask, "iss_max_z"
]

outside = iss_monthly.loc[
    ~disruption_mask, "iss_max_z"
]

# Observed difference in means
observed_difference = (
    disruption.mean() - outside.mean()
)

print("=== ISS DISRUPTION-WINDOW COMPARISON ===")

print("\nDisruption-and-response months:")
print(
    iss_monthly.loc[
        disruption_mask,
        ["month", "iss_max_z"]
    ].to_string(index=False)
)

print("\nOutside-period months:")
print(
    iss_monthly.loc[
        ~disruption_mask,
        ["month", "iss_max_z"]
    ].to_string(index=False)
)

print("\nNumber of disruption months:", len(disruption))
print("Number of outside months:", len(outside))

print("\nDisruption-period mean iss_max_z:",
      disruption.mean())

print("Outside-period mean iss_max_z:",
      outside.mean())

print("\nObserved mean difference (disruption - outside):",
      observed_difference)

=== ISS DISRUPTION-WINDOW COMPARISON ===

Disruption-and-response months:
     month  iss_max_z
2025-03-01   1.930034
2025-04-01   1.194371
2025-05-01   1.980964

Outside-period months:
     month  iss_max_z
2024-01-01   1.330143
2024-02-01   2.376912
2024-03-01   1.823902
2024-04-01   3.379372
2024-05-01   1.738498
2024-06-01   0.277416
2024-07-01   0.815506
2024-08-01   0.841145
2024-09-01  -0.012881
2024-10-01  -0.543342
2024-11-01   1.151723
2024-12-01   2.298593
2025-01-01   1.635769
2025-02-01   0.322483
2025-06-01   0.875310
2025-07-01   0.875310
2025-08-01   0.540129
2025-09-01   0.654884
2025-10-01   0.213487
2025-11-01  -0.198155
2025-12-01   0.911511

Number of disruption months: 3
Number of outside months: 21

Disruption-period mean iss_max_z: 1.7017895952818953
Outside-period mean iss_max_z: 1.014653085497076

Observed mean difference (disruption - outside): 0.6871365097848194


In [11]:
# ============================================================
# EXACT PERMUTATION TEST
# ISS disruption window vs outside period
# Same data and groups as the original t-test
# ============================================================

from itertools import combinations
import numpy as np
import pandas as pd

# Observed data
values = iss_monthly["iss_max_z"].to_numpy()

n_total = len(values)
n_disruption = len(disruption)

# Observed difference:
# disruption mean - outside mean
observed_diff = (
    disruption.mean() - outside.mean()
)

# Generate all possible assignments of 3 observations
# to the disruption group
permutation_diffs = []

for idx in combinations(range(n_total), n_disruption):

    idx = np.array(idx)

    perm_disruption = values[idx]

    # Remaining observations form the comparison group
    mask = np.ones(n_total, dtype=bool)
    mask[idx] = False
    perm_outside = values[mask]

    diff = (
        perm_disruption.mean()
        - perm_outside.mean()
    )

    permutation_diffs.append(diff)

permutation_diffs = np.array(permutation_diffs)

# Two-sided exact permutation p-value
p_value = (
    np.sum(
        np.abs(permutation_diffs)
        >= np.abs(observed_diff)
    )
    / len(permutation_diffs)
)

print("=== EXACT ISS PERMUTATION TEST ===")

print("\nObserved disruption-period mean:",
      disruption.mean())

print("Observed outside-period mean:",
      outside.mean())

print("\nObserved mean difference:",
      observed_diff)

print("\nTotal observations:", n_total)
print("Disruption observations:", n_disruption)
print("Outside observations:", len(outside))

print("\nNumber of exact permutations:",
      len(permutation_diffs))

print("\nTwo-sided permutation p-value:",
      p_value)

# Save the permutation distribution for later validation/plotting
permutation_results = pd.DataFrame({
    "permuted_mean_difference": permutation_diffs
})

permutation_results.to_csv(
    "/content/ISS_permutation_distribution.csv",
    index=False
)

print("\nSaved:")
print("/content/ISS_permutation_distribution.csv")

=== EXACT ISS PERMUTATION TEST ===

Observed disruption-period mean: 1.7017895952818953
Observed outside-period mean: 1.014653085497076

Observed mean difference: 0.6871365097848194

Total observations: 24
Disruption observations: 3
Outside observations: 21

Number of exact permutations: 2024

Two-sided permutation p-value: 0.23468379446640317

Saved:
/content/ISS_permutation_distribution.csv


In [12]:
# ============================================================
# SAVE FINAL ISS PERMUTATION TEST RESULT
# ============================================================

iss_permutation_summary = pd.DataFrame({
    "metric": [
        "analysis_period",
        "disruption_window",
        "metric",
        "disruption_n",
        "outside_n",
        "disruption_mean_iss_max_z",
        "outside_mean_iss_max_z",
        "observed_mean_difference",
        "number_of_exact_permutations",
        "test_type",
        "permutation_p_value"
    ],
    "value": [
        "2024-01 to 2025-12",
        "2025-03 to 2025-05",
        "iss_max_z",
        len(disruption),
        len(outside),
        disruption.mean(),
        outside.mean(),
        observed_diff,
        len(permutation_diffs),
        "Two-sided exact permutation test",
        p_value
    ]
})

summary_path = "/content/ISS_PERMUTATION_TEST_FINAL.csv"

iss_permutation_summary.to_csv(
    summary_path,
    index=False
)

print("Saved:")
print(summary_path)

print("\nFinal permutation-test result:")
print(iss_permutation_summary.to_string(index=False))

Saved:
/content/ISS_PERMUTATION_TEST_FINAL.csv

Final permutation-test result:
                      metric                            value
             analysis_period               2024-01 to 2025-12
           disruption_window               2025-03 to 2025-05
                      metric                        iss_max_z
                disruption_n                                3
                   outside_n                               21
   disruption_mean_iss_max_z                          1.70179
      outside_mean_iss_max_z                         1.014653
    observed_mean_difference                         0.687137
number_of_exact_permutations                             2024
                   test_type Two-sided exact permutation test
         permutation_p_value                         0.234684


In [13]:
import os

# ============================================================
# FINAL FILE VERIFICATION — NO RECALCULATION
# ============================================================

files_to_check = [
    "/content/AMPI_24_month_input_CORRECTED_FINAL.csv",
    "/content/AMPI_FINAL_CORRECTED_SARIMA.csv",
    "/content/AMPI_FINAL_CORRECTED_SARIMA_SUMMARY.csv",
    "/content/ISS_PERMUTATION_TEST_FINAL.csv",
    "/content/ISS_permutation_distribution.csv"
]

print("=== FINAL FILE CHECK ===\n")

all_found = True

for file_path in files_to_check:
    if os.path.exists(file_path):
        size_kb = os.path.getsize(file_path) / 1024
        print(f"✓ FOUND  | {os.path.basename(file_path)} | {size_kb:.2f} KB")
    else:
        print(f"✗ MISSING | {os.path.basename(file_path)}")
        all_found = False

print("\n" + "=" * 55)

if all_found:
    print("ALL 5 FILES ARE PRESENT ✓")
else:
    print("ONE OR MORE FILES ARE MISSING — DO NOT UPLOAD YET.")

=== FINAL FILE CHECK ===

✓ FOUND  | AMPI_24_month_input_CORRECTED_FINAL.csv | 1.88 KB
✓ FOUND  | AMPI_FINAL_CORRECTED_SARIMA.csv | 4.91 KB
✓ FOUND  | AMPI_FINAL_CORRECTED_SARIMA_SUMMARY.csv | 0.23 KB
✓ FOUND  | ISS_PERMUTATION_TEST_FINAL.csv | 0.37 KB
✓ FOUND  | ISS_permutation_distribution.csv | 39.33 KB

ALL 5 FILES ARE PRESENT ✓
